In [37]:
# Import Necessary Packages/Modules
import time, warnings
import numpy as np
import pandas as pd
from scipy import stats
from skeLCS import eLCS   # pip install scikit-eLCS
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import SelectFromModel
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import balanced_accuracy_score, f1_score, confusion_matrix

#Reusable constants.  You can change these to match your own data and preferences.
DATA_FILE  = "medical_insurance_cleaned_v2.csv"   # output of Task 3
SEED       = 42
N_SPLITS   = 3      # same 3-fold CV as the baseline
N_REPEATS  = 1      # raise to 5 for a meaningful significance test (15 scores per model, much slower)
N_FEATURES = 4      # features kept by the selector
N_JOBS     = -1     # folds run in parallel; set to 1 if you hit memory/pickling problems

In [38]:
# Define columns that may introduce data leakage
#These values are derived from the target variable and should not be used as features in the model to avoid data leakage
LEAKAGE_COLS = ["is_high_risk", "annual_premium", "monthly_premium", "annual_medical_cost",
                "claims_count", "avg_claim_amount", "total_claims_paid"]

#converting the risk_score into 3 bands (low, medium, high) using quantiles and storing it in a new column called risk_band
df = pd.read_csv(DATA_FILE, keep_default_na=False, na_values=[""])   # keeps "None" in alcohol_freq as a category
y = pd.qcut(df["risk_score"], 3, labels=False).values                # 0 = low, 1 = medium, 2 = high
BAND_NAMES = {0: "Low", 1: "Medium", 2: "High"}

# Prepare the feature matrix X by dropping the leakage columns, risk_score, and person_id. Convert categorical columns to integer codes for model compatibility.
X_df = df.drop(columns=LEAKAGE_COLS + ["risk_score", "person_id"])
for c in X_df.select_dtypes(exclude="number"):
    X_df[c] = X_df[c].astype("category").cat.codes                   # text -> integer codes
feature_names = list(X_df.columns)
X = X_df.values.astype(float)
print("X:", X.shape, "| class shares:", np.round(np.bincount(y) / len(y), 3))

X: (96713, 45) | class shares: [0.349 0.328 0.322]


In [39]:
# Define a feature selector function that uses a shallow decision tree to select the N_FEATURES most important features. This function will be used in the pipeline to ensure that feature selection is performed on each training fold without leaking information from the test fold.
def selector():
    # keeps the N_FEATURES most important columns according to a shallow tree.
    # Inside a Pipeline it is refit on each training fold only, so the test fold never influences selection.
    return SelectFromModel(DecisionTreeClassifier(max_depth=5, random_state=SEED),
                           max_features=N_FEATURES, threshold=-np.inf)
class ScikitELCSWrapper(BaseEstimator, ClassifierMixin):
    def __init__(self, N=1000, learning_iterations=10000, do_correct_set_subsumption=False, random_state=None):
        self.N = N
        self.learning_iterations = learning_iterations
        self.do_correct_set_subsumption = do_correct_set_subsumption
        self.random_state = random_state

    def fit(self, X, y):
        y_int = y.astype(int)
        self.model_ = eLCS(
            N=self.N,
            learning_iterations=self.learning_iterations,
            do_correct_set_subsumption=self.do_correct_set_subsumption,
            random_state=self.random_state
        )
        self.model_.fit(X, y_int)
        self.classes_ = np.unique(y_int)
        return self

    def predict(self, X):
        if hasattr(self.model_, "predict"):
            return self.model_.predict(X)
        elif hasattr(self.model_, "predict_label"):
            return self.model_.predict_label(X)
        else:
            raise AttributeError("eLCS instance has no valid prediction method.")

configs = {
    "A. Original eLCS, preprocessed data (45 features)": Pipeline([
        ("elcs", ScikitELCSWrapper(random_state=SEED))
    ]),
    "B. + feature selection": Pipeline([
        ("select", selector()),
        ("elcs", ScikitELCSWrapper(random_state=SEED))
    ]),
    "C. + MinMax scaling": Pipeline([
        ("select", selector()), 
        ("scale", MinMaxScaler()),
        ("elcs", ScikitELCSWrapper(random_state=SEED))
    ]),
    "D. + correct-set subsumption": Pipeline([
        ("select", selector()), 
        ("scale", MinMaxScaler()),
        ("elcs", ScikitELCSWrapper(random_state=SEED, do_correct_set_subsumption=True))
    ]),
    "E. Improved (+ 20k iterations, N=1500)": Pipeline([
        ("select", selector()), 
        ("scale", MinMaxScaler()),
        ("elcs", ScikitELCSWrapper(
            random_state=SEED, 
            do_correct_set_subsumption=True,
            learning_iterations=20000, 
            N=1500
        ))
    ]),
}
IMPROVED = "E. Improved (+ 20k iterations, N=1500)"
BASELINE = "A. Original eLCS, preprocessed data (45 features)"

In [40]:
# Run cross-validation for each configuration and collect scores and execution times. The scoring metrics used are balanced accuracy and macro F1 score. The results are printed for each configuration, showing the mean and standard deviation of the balanced accuracy, the mean macro F1 score, and the time taken for cross-validation.
cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=SEED)
scoring = {"bal_acc": "balanced_accuracy", "f1_macro": "f1_macro"}  #scoring metrics to be used for cross-validation

scores, times = {}, {} #containers to hold the cross-validation scores and execution times for each configuration
for name, pipe in configs.items():
    t0 = time.time()
    res = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=N_JOBS)
    scores[name] = res
    times[name] = time.time() - t0
    print(f"{name}\n   balanced acc {res['test_bal_acc'].mean():.3f} +/- {res['test_bal_acc'].std():.3f}"
          f" | macro-F1 {res['test_f1_macro'].mean():.3f} | {times[name]:.0f}s")

A. Original eLCS, preprocessed data (45 features)
   balanced acc 0.542 +/- 0.021 | macro-F1 0.511 | 48s
B. + feature selection
   balanced acc 0.787 +/- 0.034 | macro-F1 0.767 | 29s
C. + MinMax scaling
   balanced acc 0.787 +/- 0.034 | macro-F1 0.767 | 25s
D. + correct-set subsumption
   balanced acc 0.828 +/- 0.028 | macro-F1 0.818 | 31s
E. Improved (+ 20k iterations, N=1500)
   balanced acc 0.866 +/- 0.011 | macro-F1 0.863 | 42s


In [41]:
summary = pd.DataFrame({
    "Balanced accuracy (mean)": {n: r["test_bal_acc"].mean() for n, r in scores.items()},
    "Std across folds":         {n: r["test_bal_acc"].std()  for n, r in scores.items()},
    "Macro-F1 (mean)":          {n: r["test_f1_macro"].mean() for n, r in scores.items()},
    "Time (s)":                 times,
}).round(3)
summary

,Balanced accuracy (mean),Std across folds,Macro-F1 (mean),Time (s)
"A. Original eLCS, preprocessed data (45 features)",0.542,0.021,0.511,48.395
B. + feature selection,0.787,0.034,0.767,28.858
C. + MinMax scaling,0.787,0.034,0.767,24.520
D. + correct-set subsumption,0.828,0.028,0.818,30.627
"E. Improved (+ 20k iterations, N=1500)",0.866,0.011,0.863,42.222


In [ ]:
#Compare scores with the Nadeau-Bengio test
def nadeau_bengio(a, b, n_train, n_test):
    d = np.asarray(a) - np.asarray(b)  #difference between the two sets of scores
    k = len(d)  #number of folds (or number of paired scores)
    var = d.var(ddof=1)  #variance of the differences
    if var == 0: #if the variance is zero, return NaN for both t-statistic and p-value
        return np.nan, np.nan #return NaN for both t-statistic and p-value
    t = d.mean() / np.sqrt((1 / k + n_test / n_train) * var)
    return t, 2 * stats.t.sf(abs(t), k - 1)

n_test  = int(len(y) / N_SPLITS)
n_train = len(y) - n_test
base = scores[BASELINE]["test_bal_acc"]

rows = []
for name, r in scores.items():
    if name == BASELINE:
        continue
    s = r["test_bal_acc"]
    try:
        w_p = stats.wilcoxon(s, base).pvalue
    except ValueError:# all differences zero
        w_p = np.nan
    t, nb_p = nadeau_bengio(s, base, n_train, n_test)
    rows.append({"Configuration vs baseline": name, "Mean diff": s.mean() - base.mean(),
                 "Wilcoxon p": w_p, "Nadeau-Bengio p": nb_p})
pd.DataFrame(rows).round(4)

,Configuration vs baseline,Mean diff,Wilcoxon p,Nadeau-Bengio p
0,B. + feature selection,0.2444,0.25,0.0440
1,C. + MinMax scaling,0.2444,0.25,0.0440
2,D. + correct-set subsumption,0.2854,0.25,0.0352
3,"E. Improved (+ 20k iterations, N=1500)",0.3241,0.25,0.0059


In [ ]:
# Hold-out evaluation of the final model on a separate test set. 
from sklearn.metrics import classification_report
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

# Fit the final model using the best configuration (IMPROVED) on the training set and evaluate it on the hold-out test set. 
elcs  = final.named_steps["elcs"]
pred  = final.predict(Xte)

print(f"Hold-out balanced accuracy: {balanced_accuracy_score(yte, pred):.3f}")
print(f"Hold-out macro-F1:          {f1_score(yte, pred, average='macro'):.3f}")

# Point directly to the internal model_ attribute:
print(f"Training accuracy of final rule set: {elcs.model_.get_final_accuracy():.3f}")
print(f"Training instance coverage:          {elcs.model_.get_final_instance_coverage():.1%}")

print("\nConfusion matrix (rows = true, cols = predicted):\n", confusion_matrix(yte, pred))
print("\n", classification_report(yte, pred, target_names=[BAND_NAMES[i] for i in range(3)], digits=3))
print("Selected features:", [f for f, keep in zip(feature_names, final.named_steps["select"].get_support()) if keep])

Hold-out balanced accuracy: 0.854
Hold-out macro-F1:          0.848
Training accuracy of final rule set: 0.855
Training instance coverage:          100.0%

Confusion matrix (rows = true, cols = predicted):
 [[6759    0    1]
 [1687 3967  696]
 [  41  351 5841]]

               precision    recall  f1-score   support

         Low      0.796     1.000     0.887      6760
      Medium      0.919     0.625     0.744      6350
        High      0.893     0.937     0.915      6233

    accuracy                          0.856     19343
   macro avg      0.869     0.854     0.848     19343
weighted avg      0.868     0.856     0.849     19343

Selected features: ['age', 'bmi', 'smoker', 'chronic_count']


In [44]:
# Pruning function to filter rules based on accuracy and numerosity
def prune(model, min_acc=0.9, min_num=2):
    target_model = getattr(model, "model_", model)
    # skeLCS uses popSet (camelCase)
    return [r for r in target_model.population.popSet if r.accuracy >= min_acc and r.numerosity >= min_num]

# Store a copy of the original rule set
full_pop = list(elcs.model_.population.popSet)

print("Rules before compaction (macro):", len(full_pop),
      "| micro (sum of numerosity):", sum(r.numerosity for r in full_pop))

for min_acc, min_num in [(0.0, 1), (0.8, 2), (0.9, 2), (0.9, 5)]:
    kept = prune(elcs, min_acc, min_num)
    elcs.model_.population.popSet = kept
    p = final.predict(Xte)
    print(f"min_acc={min_acc}, min_num={min_num}: {len(kept):4d} rules | "
          f"balanced acc {balanced_accuracy_score(yte, p):.3f}")

# Restore the full rule set
elcs.model_.population.popSet = full_pop

Rules before compaction (macro): 693 | micro (sum of numerosity): 1500
min_acc=0.0, min_num=1:  693 rules | balanced acc 0.854
min_acc=0.8, min_num=2:  227 rules | balanced acc 0.856
min_acc=0.9, min_num=2:  153 rules | balanced acc 0.766
min_acc=0.9, min_num=5:   53 rules | balanced acc 0.717


In [45]:
selected = [f for f, keep in zip(feature_names, final.named_steps["select"].get_support()) if keep]
scaler = final.named_steps["scale"]

def describe(rule):
    parts = []
    # skeLCS uses specifiedAttList and condition
    for idx, cond in zip(rule.specifiedAttList, rule.condition):
        lo_s, rng = scaler.data_min_[idx], scaler.data_range_[idx]
        if isinstance(cond, (list, tuple, np.ndarray)):
            parts.append(f"{selected[idx]} in [{cond[0]*rng + lo_s:.2f}, {cond[1]*rng + lo_s:.2f}]")
        else:
            parts.append(f"{selected[idx]} = {cond*rng + lo_s:.2f}")
    return " AND ".join(parts) if parts else "DEFAULT (Any)"

# Use elcs.model_.population.popSet instead of elcs.population.pop_set
target_model = getattr(elcs, "model_", elcs)
top = sorted(target_model.population.popSet, key=lambda r: r.numerosity * r.fitness, reverse=True)[:10]

rule_table = pd.DataFrame([{
    "IF": describe(r), 
    "THEN": BAND_NAMES[int(r.phenotype)],
    "accuracy": round(r.accuracy, 3), 
    "fitness": round(r.fitness, 3), 
    "numerosity": r.numerosity
} for r in top])

pd.set_option("display.max_colwidth", 120)
rule_table

,IF,THEN,accuracy,fitness,numerosity
0,"age in [40.17, 47.81] AND chronic_count = 0.00 AND smoker = 2.00",Low,1.000,1.0,16
1,chronic_count = 4.00,High,1.000,1.0,15
2,"age in [8.48, 27.20] AND chronic_count = 1.00 AND smoker = 1.00",Low,1.000,1.0,15
3,"age in [81.81, 111.52] AND chronic_count = 1.00",High,1.000,1.0,14
4,chronic_count = 3.00,High,0.979,0.9,15
5,"age in [42.37, 77.63] AND chronic_count = 2.00",High,1.000,1.0,13
6,"age in [14.19, 27.20] AND smoker = 2.00 AND chronic_count = 1.00",Low,1.000,1.0,13
7,"smoker = 0.00 AND age in [37.74, 63.04] AND chronic_count = 1.00",High,1.000,1.0,12
8,"age in [-12.33, 47.81] AND chronic_count = 0.00 AND smoker = 1.00",Low,1.000,1.0,11
9,"chronic_count = 2.00 AND age in [42.37, 85.42]",High,1.000,1.0,11
